# PFC · Avaliação dos modelos em GPU de nuvem (Colab / Kaggle)

Roda o **mesmo** `pfc-avaliar` do repositório, com o **mesmo** Ollama, os **mesmos** modelos e a **mesma** quantização — só que numa T4 de 16 GB, onde todos cabem inteiros na VRAM.

O que muda em relação à estação do Cap. 3 é só o hardware, e isso fica registrado no `manifesto.json` de cada modelo (o avaliador anota `nvidia-smi`, versões e hash do dataset). No texto, o ambiente é descrito **como ele é**: GPU T4 (Colab) para as rodadas completas; RTX 3050 para o subconjunto de validação.

**Antes de começar:** `Ambiente de execução → Alterar tipo → GPU (T4)`. Depois execute as células em ordem. Uma sessão gratuita dura até 12 h; a rodada completa deve levar ~3 h.

> Este notebook **não** usa nenhum serviço de LLM em nuvem: o Ollama roda dentro desta máquina virtual, com os pesos baixados aqui. Só a GPU é alugada.

In [ ]:
# 1) GPU disponível?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2) Instala o Ollama e sobe o servidor em segundo plano
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, os, urllib.request, json
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_KEEP_ALIVE'] = '30m'
servidor = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        v = json.load(urllib.request.urlopen('http://localhost:11434/api/version', timeout=3))['version']
        print('Ollama', v, 'no ar'); break
    except Exception:
        time.sleep(2)

In [ ]:
# 3) Envie o arquivo dist/pfc_busca_colab.zip (gerado no seu PC) quando o botão aparecer
from google.colab import files
enviado = files.upload()
nome = next(iter(enviado))
!rm -rf /content/pfc_busca && unzip -q -o "$nome" -d /content && ls /content/pfc_busca
%cd /content/pfc_busca
!pip install -q -e . 2>&1 | tail -1
!pfc-dataset && echo dataset ok

In [ ]:
# 4) Baixa os quatro modelos (~19 GB; 5-10 min no Colab)
MODELOS = ['qwen3:4b-instruct-2507-q4_K_M', 'gemma4:e4b-it-qat', 'gemma4:e2b-it-qat', 'mistral-nemo:12b']
for m in MODELOS:
    !ollama pull {m} 2>&1 | tail -1
!ollama list

In [ ]:
# 5) Rodada F6: 310 consultas x 3 repetições por modelo, data de referência FIXA
#    (retomável: se a sessão cair, rode esta célula de novo com o mesmo HOJE)
import datetime
HOJE = datetime.date.today().isoformat()   # anote este valor
REPETICOES = 3
print('hoje =', HOJE)
for m in MODELOS:
    print('\n==', m, '==')
    !pfc-avaliar --modelo {m} --repeticoes {REPETICOES} --hoje {HOJE} --sem-sql 2>&1 | grep -v '^  [✓·×]' | tail -12
    !ollama stop {m}

In [ ]:
# 6) Consolida (tabelas do Cap. 5, figuras, estatística) e baixa tudo
!pfc-relatorio --modelos {','.join(MODELOS)}
!cd /content && zip -q -r resultados_colab.zip pfc_busca/results
files.download('/content/resultados_colab.zip')

## De volta ao PC

Descompacte `resultados_colab.zip` sobre `pfc_busca/results/` e rode `pfc-relatorio` localmente se quiser regerar as figuras. Cada `results/<modelo>/manifesto.json` diz onde e como aquela rodada aconteceu — é isso que entra no Cap. 5.

**Kaggle** funciona igual (Settings → Accelerator → GPU T4 x2; 30 h/semana): troque a célula 3 por um *dataset* do Kaggle com o zip, e a célula 6 por salvar em `/kaggle/working/`.